# Embedding Models Comparision

In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
import time
from tqdm import tqdm
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
models = [
    "intfloat/multilingual-e5-small",
    "microsoft/harrier-oss-v1-270m",
    "Qwen/Qwen3-Embedding-0.6B",
]

In [5]:
test_queries = [
    "sürücü belgesi bulundurmamanın cezası",
    "araç sürücüleri trafik kazalarında yapması gerekenler",
]

In [6]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

## Speed

In [7]:
runs = 10
models_speed = {model: [] for model in models}

for model_name in tqdm(models):
    # Warm-up
    model = SentenceTransformer(model_name, token=hf_token)
    model.encode(test_queries)

    for _ in range(runs):
        t0 = time.perf_counter()
        model.encode(test_queries, batch_size=32)
        models_speed[model_name].append(time.perf_counter() - t0)

pd.DataFrame(models_speed).describe()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 6918.83it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}
100%|██████████| 3/3 [00:32<00:00, 10.70s/it]


,intfloat/multilingual-e5-small,microsoft/harrier-oss-v1-270m,Qwen/Qwen3-Embedding-0.6B
count,10.000000,10.000000,10.000000
mean,0.018019,0.073284,0.089081
std,0.000927,0.003894,0.009590
min,0.017151,0.067895,0.082314
25%,0.017369,0.071062,0.083289
50%,0.017750,0.072014,0.085578
75%,0.018322,0.076621,0.091542
max,0.020256,0.079234,0.113987


## Memory

In [8]:
def get_model_disk_size_mb(model_name: str) -> float:
    folder = os.path.expanduser(
        f"~/.cache/huggingface/hub/models--{model_name.replace('/', '--')}"
    )
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(folder)
        for f in files
    )
    return round(total / 1024 / 1024, 1)


for model_name in models:
    print(f"{model_name.split('/')[-1]}: {get_model_disk_size_mb(model_name)} MB")

multilingual-e5-small: 4328.0 MB
harrier-oss-v1-270m: 1088.7 MB
Qwen3-Embedding-0.6B: 3454.7 MB


## Retrieval Quality

In [9]:
test_pairs = [
    {
        "query": "Araç muayene raporu",
        "relevant": "Bu Kanunun 35 inci maddesiyle yetkilendirilen kuruluşlarca araç muayene raporu tanzim edilebilmesi için araç tescil belgesi veya sahiplik belgesi ile zorunlu mali sorumluluk sigortasının ibrazı zorunludur.",
        "irrelevant": "Uyuşturucu veya uyarıcı maddeleri almış olan sürücüler ile alkollü olan sürücülerin karayolunda araç sürmeleri yasaktır.",
    },
    {
        "query": "Araçların ışıklandırılması",
        "relevant": "Karayolunda trafiğe çıkan bütün araçların, nicelik ve nitelikleri yönetmelikte belirtilen şartlara uygun ışık donanımı bulundurmaları zorunludur.",
        "irrelevant": "Araçlarda ses, müzik, görüntü ve haberleşme cihazları yönetmelikte gösterilen şartlara uygun olarak ve kamunun rahat ve huzurunu bozmayacak şekilde kullanılabilir.",
    },
]


def benchmark_quality(model, pairs: list[dict]) -> float:
    scores = []
    for pair in pairs:
        q_emb = model.encode([pair["query"]])
        r_emb = model.encode([pair["relevant"]])
        i_emb = model.encode([pair["irrelevant"]])

        rel_score = cosine_similarity(q_emb, r_emb)[0][0]
        irr_score = cosine_similarity(q_emb, i_emb)[0][0]

        scores.append(round(rel_score - irr_score, 5))

    return np.mean(scores)


models_quality = {model: [] for model in models}

for model_name in models:
    model = SentenceTransformer(model_name)
    print(f"{model_name} avg quality score: {benchmark_quality(model, test_pairs)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7652.09it/s]


intfloat/multilingual-e5-small avg quality score: 0.03882499784231186


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 7026.13it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


microsoft/harrier-oss-v1-270m avg quality score: 0.09552499651908875


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4615.01it/s]


Qwen/Qwen3-Embedding-0.6B avg quality score: 0.1545799970626831


## Final Decision

When comparing the models with speed, memory and relevance, there is no hat-trick for a model. So, our decision is depend on us in that moment. I will go forward with intfloat's model because of its memory (because of my setup).